# PySpark Intro
Connect to the Spark cluster and run basic DataFrame operations.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("spark_intro")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("App UI:", spark.sparkContext.uiWebUrl)

Spark version: 3.5.1
App UI: http://31a596a6a0bf:4040


In [2]:
data = [
    {"name": "Alice",   "dept": "Engineering", "salary": 95000},
    {"name": "Bob",     "dept": "Marketing",   "salary": 72000},
    {"name": "Carol",   "dept": "Engineering", "salary": 105000},
    {"name": "Dave",    "dept": "Marketing",   "salary": 68000},
    {"name": "Eve",     "dept": "Engineering", "salary": 88000},
    {"name": "Frank",   "dept": "HR",          "salary": 61000},
]

df = spark.createDataFrame(data)
df.printSchema()
df.show()

root
 |-- dept: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)

+-----------+-----+------+
|       dept| name|salary|
+-----------+-----+------+
|Engineering|Alice| 95000|
|  Marketing|  Bob| 72000|
|Engineering|Carol|105000|
|  Marketing| Dave| 68000|
|Engineering|  Eve| 88000|
|         HR|Frank| 61000|
+-----------+-----+------+



In [3]:
# Filter: only Engineering employees
df.filter(df.dept == "Engineering").show()

+-----------+-----+------+
|       dept| name|salary|
+-----------+-----+------+
|Engineering|Alice| 95000|
|Engineering|Carol|105000|
|Engineering|  Eve| 88000|
+-----------+-----+------+



In [4]:
# Aggregation: average salary per department
from pyspark.sql.functions import avg, count

df.groupBy("dept").agg(
    count("name").alias("headcount"),
    avg("salary").alias("avg_salary")
).orderBy("dept").show()

+-----------+---------+----------+
|       dept|headcount|avg_salary|
+-----------+---------+----------+
|Engineering|        3|   96000.0|
|         HR|        1|   61000.0|
|  Marketing|        2|   70000.0|
+-----------+---------+----------+



In [5]:
# Select & derive: name + salary above median flag
from pyspark.sql.functions import col, when

median_salary = 80000

df.select(
    col("name"),
    col("salary"),
    when(col("salary") >= median_salary, "above").otherwise("below").alias("vs_median")
).show()

+-----+------+---------+
| name|salary|vs_median|
+-----+------+---------+
|Alice| 95000|    above|
|  Bob| 72000|    below|
|Carol|105000|    above|
| Dave| 68000|    below|
|  Eve| 88000|    above|
|Frank| 61000|    below|
+-----+------+---------+



In [6]:
spark.stop()